Forecast del caudal de los ríos de una zona geográfica  

### Diccionario de Datos: Dataset de Predicción (Forecast)

| Columna | Descripción | Unidad |
| :--- | :--- | :--- |
| **`valid_time`** | **Fecha de impacto.** Representa el momento exacto en el futuro para el cual se hace la predicción. Es la fecha que se compara con la realidad histórica. | `datetime` |
| **`forecast_reference_time`** | **Fecha de emisión.** Momento en el que el modelo generó la predicción (el "ahora" del sistema de alerta). | `datetime` |
| **`forecast_period`** | **Horizonte de predicción (Leadtime).** Desfase temporal entre la emisión y el impacto. Indica la antelación del aviso. | `timedelta` (h) |
| **`caudal_predicho_m3s`** | **Caudal previsto.** Volumen de agua estimado que circulará por el cauce en el `valid_time`. | $m^3/s$ |
| **`surface`** | **Índice de humedad superficial.** Nivel de saturación previsto para la capa superior del suelo en la fecha futura. | Adimensional (0-1) |
| **`latitude`** | Latitud del punto geográfico (píxel de la red fluvial). | Grados decimales |
| **`longitude`** | Longitud del punto geográfico (píxel de la red fluvial). | Grados decimales |
| **`number`** | **ID del miembro.** Identificador de la simulación (0 para el control principal). | Entero |

In [ ]:
import os

# Configuramos el mensajero para que vaya al portal de Emergencias
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write('url: https://ewds.climate.copernicus.eu/api\n')
    f.write('key: b1c8ed63-d58a-4ca0-b6a8-0068ce339221')

In [ ]:
import cdsapi

client = cdsapi.Client()

# Parámetros obtenidos de la web (validados)
dataset = "cems-glofas-forecast"
request = {
    "system_version": ["operational"],
    "hydrological_model": ["lisflood"],
    "product_type": ["control_forecast"],
    "variable": "river_discharge_in_the_last_24_hours",
    "year": ["2025"],
    "month": ["03"],
    "day": ["01", "02", "03", "04", "05"], # Solo 5 días de "emisión"
    "leadtime_hour": ["24", "48", "72"], # Solo 3 días de "vista al futuro"
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [43.51, -4.85, 42.76, -3.15]
}

target_file = "caudal_forecast_2025.nc"

try:
    print("Iniciando descarga de Forecast Operacional...")
    client.retrieve(dataset, request).download(target_file)
    print(f"¡Éxito! Archivo guardado como: {target_file}")
except Exception as e:
    print(f"Error en la descarga: {e}")

Iniciando descarga de Forecast Operacional...


2026-05-03 19:51:39,670 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-05-03 19:51:39,671 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-05-03 19:51:39,672 INFO Request ID is 75d688eb-9ed8-4ccb-acb2-5295939b4ec9
2026-05-03 19:51:40,272 INFO status has been updated to accepted
2026-05-03 19:52:19,510 INFO status has been updated to running
2026-05-03 20:02:13,959 INFO status has been updated to successful
                                                                                         

¡Éxito! Archivo guardado como: caudal_forecast_2025.nc


In [ ]:
import xarray as xr
import pandas as pd
import os
from datetime import timedelta

# --- CONFIGURACIÓN ---
input_file = "caudal_forecast_2025.nc" # El nombre que pusiste en la descarga
output_csv = "copernicus_caudal_forecasts.csv"

try:
    print(f"Procesando predicciones de: {input_file}")
    
    # 1. Abrir el dataset
    # Usamos decode_coords='all' para que lat/lon se gestionen bien
    with xr.open_dataset(input_file, engine='h5netcdf') as ds:
        
        # 2. Convertir a DataFrame
        df = ds.to_dataframe().reset_index()
        
        # 3. Limpiar valores nulos (píxeles donde no hay río)
        # En el forecast la columna suele llamarse 'dis24'
        col_caudal = [c for c in df.columns if 'dis' in c][0]
        df = df.dropna(subset=[col_caudal])
        
        # 4. CÁLCULO CRÍTICO PARA TU TFM: La fecha de impacto
        # 'time' es cuando se emite el forecast (ej: 01-Marzo)
        # 'step' es el horizonte (ej: 48 horas)
        # Creamos 'fecha_prevista' (ej: 03-Marzo)
        if 'step' in df.columns:
            df['fecha_prevista'] = df['time'] + df['step']
        
        # 5. Renombrar para que sea legible
        df = df.rename(columns={col_caudal: 'caudal_predicho_m3s'})

        # 6. Guardar el resultado
        df.to_csv(output_csv, index=False)
        
    print(f"--- ¡CONVERSIÓN DE FORECAST EXITOSA! ---")
    print(f"Archivo guardado: {output_csv}")
    print(f"Columnas generadas: {list(df.columns)}")
    display(df.head())

except Exception as e:
    print(f"Error en la conversión: {e}")
    print("Asegúrate de tener instalado: pip install h5netcdf xarray pandas")

Procesando predicciones de: caudal_forecast_2025.nc
--- ¡CONVERSIÓN DE FORECAST EXITOSA! ---
Archivo guardado: predicciones_caudal_tfm.csv
Columnas generadas: ['forecast_period', 'forecast_reference_time', 'latitude', 'longitude', 'caudal_predicho_m3s', 'number', 'surface', 'valid_time']


,forecast_period,forecast_reference_time,latitude,longitude,caudal_predicho_m3s,number,surface,valid_time
18,1 days,2025-03-01,43.475,356.075,0.18750,0,0.0,2025-03-02
19,1 days,2025-03-01,43.475,356.125,0.34375,0,0.0,2025-03-02
20,1 days,2025-03-01,43.475,356.175,0.21875,0,0.0,2025-03-02
22,1 days,2025-03-01,43.475,356.275,0.15625,0,0.0,2025-03-02
23,1 days,2025-03-01,43.475,356.325,0.31250,0,0.0,2025-03-02
